# Aggregations

In [1]:
import os
os.environ["JAVA_HOME"] = "/Library/Java/JavaVirtualMachines/temurin-23.jdk/Contents/Home"  # or wherever your JDK 17/21 lives
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

In [2]:
from pyspark.sql import SparkSession
spark = (
    SparkSession.builder.appName("Local Spark Session")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "spark-warehouse")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "8g")
    .enableHiveSupport()
    .getOrCreate()
)
print(spark.sparkContext.uiWebUrl)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/02 20:32:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


http://127.0.0.1:4040


In [3]:
df = (
    spark
    .read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("/Users/jcrmendes/Projects/jcrmendes_projects/T-Shaped-Engineer/spark/data/retail-data/all/online-retail-dataset.csv")
    .coalesce(5)
)
df.cache()
df.createOrReplaceTempView("dfTable")

In [ ]:
# this is calculated over the entire dataframe. The count is an action, but it can also be used as a lazy evaluation
df.count()

541909

## Aggregation functions
### Count

In [5]:
import pyspark.sql.functions as F

# Important gotcha. When doing a count of a entire Dataframe, the count will also include NULL rows.
# But when we count on a specific column, like we do here, the null values are skipped and not considered.
df.select(F.count("StockCode")).show()

# We can also get the count distinct
df.select(F.count_distinct("StockCode")).show()

# When handling very big dataframes, sometimes we don't need the exact distinct number and an approx is ok. On those cases,
# we can use the approx_count_distinct
df.select(F.approxCountDistinct("StockCode", 0.1)).show() # The 0.1 represents the maximum estimation error allowed

+----------------+
|count(StockCode)|
+----------------+
|          541909|
+----------------+



/Users/jcrmendes/Projects/jcrmendes_projects/T-Shaped-Engineer/venv/lib/python3.13/site-packages/pyspark/sql/functions/builtin.py:6553: FutureWarning: Deprecated in 2.1, use approx_count_distinct instead.
  warnings.warn("Deprecated in 2.1, use approx_count_distinct instead.", FutureWarning)


+-------------------------+
|count(DISTINCT StockCode)|
+-------------------------+
|                     4070|
+-------------------------+

+--------------------------------+
|approx_count_distinct(StockCode)|
+--------------------------------+
|                            3364|
+--------------------------------+



In [6]:
# sumDistinct
# We can use the sum distinct to sum a distinct set of values
df.select(F.sumDistinct(F.col("Quantity"))).show()

# Although we can calculate the average by dividing sum by count, Spark also provides an easier way to calculate
(
    df
    .select(
        F.count("Quantity").alias("total_transactions"),
        F.sum("Quantity").alias("total_purchases"),
        F.avg("Quantity").alias("avg_purchases"),
        F.expr("mean(Quantity)").alias("mean_purchases")
    )
    .selectExpr(
        "total_purchases/total_transactions",
        "avg_purchases",
        "mean_purchases"
    ).show()
)

/Users/jcrmendes/Projects/jcrmendes_projects/T-Shaped-Engineer/venv/lib/python3.13/site-packages/pyspark/sql/functions/builtin.py:1836: FutureWarning: Deprecated in 3.2, use sum_distinct instead.
  warnings.warn("Deprecated in 3.2, use sum_distinct instead.", FutureWarning)


+----------------------+
|sum(DISTINCT Quantity)|
+----------------------+
|                 29310|
+----------------------+

+--------------------------------------+----------------+----------------+
|(total_purchases / total_transactions)|   avg_purchases|  mean_purchases|
+--------------------------------------+----------------+----------------+
|                      9.55224954743324|9.55224954743324|9.55224954743324|
+--------------------------------------+----------------+----------------+



## Variance and Standard Deviation

**Variance:** Variance is a measure of how spread out values are around their average. If values are very close to the mean, variance is small. IF the values are far from the mean, variance is large.

**Standard Deviation:** Standard deviation tells us the typical distance of data points from the mean. Small standard deviation, values are tightly clustered near the average. Large standard deviation, values are more spread out.

In [ ]:
(
    df
    .select(
        F.var_pop("Quantity"), # Population variance of Quantity
        F.var_samp("Quantity"), # Sample variance of Quantity
        F.stddev_pop("Quantity"), # Popular standard deviation
        F.stddev_samp("Quantity") # Sample standard deviation
    ).show()
)

+-----------------+------------------+--------------------+---------------------+
|var_pop(Quantity)|var_samp(Quantity)|stddev_pop(Quantity)|stddev_samp(Quantity)|
+-----------------+------------------+--------------------+---------------------+
|47559.30364660918| 47559.39140929887|  218.08095663447824|   218.08115785023443|
+-----------------+------------------+--------------------+---------------------+



## Skewness and Kurtosis

**Skewness:** Skewness tells you if the hill leans left or right. Skewness = 0, data is fairly balanced, positive skew > 0, long tail to the right (some very large values), negative skew < 0, long tail to the left (some very small values). Easy image, Right-skew, most people spend $20, a few spend $500. Left-skew, most exam scores are high, a few very low scores pull left.

**Kurtosis:** Kurtosis tells you how pointy or heavy-tailed the hill is. Higher kurtosis, more extreme outliers, heavier tails, often sharper center. Lower Kurtosis, fewer extreme outliers, lighter tails, flatter shape.

In [8]:
(
    df
    .select(
        F.skewness(F.col("Quantity")),
        F.kurtosis(F.col("Quantity"))
    ).show()
)

+-------------------+------------------+
| skewness(Quantity)|kurtosis(Quantity)|
+-------------------+------------------+
|-0.2640755761052702|119768.05495532964|
+-------------------+------------------+



## Correlation and Covariance
Covariance and correlation both answer:
Do these two move together, and in which direction?

### Covariance
* Positive covariance: when one goes up, the other tends to go up;
* Negative covariance: When one goes ip, the other tends to do down;
* Near zero: no clear together-movement;

Easy idea:
* Study hours and score usually have positive covariance;

### Correlation
Correlation is a scaled version of covariance, so it is easy to read.
* Range is always from -1 to +1;
* +1 means perfect same-direction line;
* -1 means perfect opposite-direction line;
* 0 means no linear relationship;

* Use covariance to know direction of joint movement;
* Use correlation to know direction plus strength in a standardized way.

In [ ]:
(
    df
    .select(
        F.corr("InvoiceNo", "Quantity"),
        F.covar_samp("InvoiceNo", "Quantity"),
        F.covar_pop("InvoiceNo", "Quantity")
    ).show()
)

26/09/02 06:54:28 ERROR Executor: Exception in task 1.0 in stage 37.0 (TID 73)
org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value 'C546529' of the type "STRING" cannot be cast to "DOUBLE" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"corr" was called from
java.base/jdk.internal.reflect.DirectMethodHandleAccessor.invoke(DirectMethodHandleAccessor.java:103)

	at org.apache.spark.sql.errors.QueryExecutionErrors$.invalidInputInCastToNumberError(QueryExecutionErrors.scala:148)
	at org.apache.spark.sql.errors.QueryExecutionErrors.invalidInputInCastToNumberError(QueryExecutionErrors.scala)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.hashAgg_subExpr_0$(Unknown Source)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.hashAgg_doConsum

NumberFormatException: [CAST_INVALID_INPUT] The value 'C536379' of the type "STRING" cannot be cast to "DOUBLE" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"corr" was called from
java.base/jdk.internal.reflect.DirectMethodHandleAccessor.invoke(DirectMethodHandleAccessor.java:103)


In [11]:
# aggregation to complex types
# We can perform aggregations not just of numerical values using formulas, we can also do them on complex types.

(
    df
    .agg(
        F.collect_set("Country"), # creates a SET
        F.collect_list("Country") # Creates a list
    ).show()
)

+--------------------+---------------------+
|collect_set(Country)|collect_list(Country)|
+--------------------+---------------------+
|[Netherlands, Sau...| [United Kingdom, ...|
+--------------------+---------------------+



In [15]:
# using the groupBy by grouping and entire dataframe
(
    df.groupBy("InvoiceNo", "CustomerId").count().show()
)

# We can also use the EXPR
(
    df
    .groupBy("InvoiceNo")
    .agg(
        F.count("Quantity").alias("quan"),
        F.expr("count(Quantity)")
    ).show()
)

# Grouping with Maps
(
    df
    .groupBy("InvoiceNo")
    .agg(
        F.expr("avg(Quantity)"),
        F.expr("stddev_pop(Quantity)")
    ).show()
)

+---------+----------+-----+
|InvoiceNo|CustomerId|count|
+---------+----------+-----+
|   536846|     14573|   76|
|   537026|     12395|   12|
|   537883|     14437|    5|
|   538068|     17978|   12|
|   538279|     14952|    7|
|   538800|     16458|   10|
|   538942|     17346|   12|
|  C539947|     13854|    1|
|   540096|     13253|   16|
|   540530|     14755|   27|
|   541225|     14099|   19|
|   541978|     13551|    4|
|   542093|     17677|   16|
|   543188|     12567|   63|
|   543590|     17377|   19|
|  C543757|     13115|    1|
|  C544318|     12989|    1|
|   544578|     12365|    1|
|   545165|     16339|   20|
|   545289|     14732|   30|
+---------+----------+-----+
only showing top 20 rows
+---------+----+---------------+
|InvoiceNo|quan|count(Quantity)|
+---------+----+---------------+
|   536596|   6|              6|
|   536938|  14|             14|
|   537252|   1|              1|
|   537691|  20|             20|
|   538041|   1|              1|
|   538184|  26

+---------+------------------+--------------------+
|InvoiceNo|     avg(Quantity)|stddev_pop(Quantity)|
+---------+------------------+--------------------+
|   536596|               1.5|  1.1180339887498947|
|   536938|33.142857142857146|  20.698023172885524|
|   537252|              31.0|                 0.0|
|   537691|              8.15|   5.597097462078001|
|   538041|              30.0|                 0.0|
|   538184|12.076923076923077|   8.142590198943392|
|   538517|3.0377358490566038|  2.3946659604837897|
|   538879|21.157894736842106|  11.811070444356483|
|   539275|              26.0|  12.806248474865697|
|   539630|20.333333333333332|  10.225241100118645|
|   540499|              3.75|  2.6653642652865788|
|   540540|2.1363636363636362|  1.0572457590557278|
|  C540850|              -1.0|                 0.0|
|   540976|10.520833333333334|   6.496760677872902|
|   541432|             12.25|  10.825317547305483|
|   541518| 23.10891089108911|  20.550782784878713|
|   541783|1

## Window functions
Window functions let you compute values per row using a dynamic subset of rows (a window/frame) related to that row.

Key idea:
* Group-by: each row belongs to exactly one group, and results are collapsed per group;
* Window function: each input row keeps its identity, and gest a computed value based on its frame;
* A row can contribute to multiple frames

In Spark, window functions are commonly used for things like rolling metrics and come in three categories:
* Ranking functions;
* Analytics functions;
* Aggregate functions;

In [9]:
# Let's add a date column
from pyspark.sql.window import Window
dfWithDate = df.withColumn("date", F.try_to_date(F.col("InvoiceDate"), "MM/d/yyyy H:mm"))
dfWithDate.createOrReplaceTempView("dfWithTable")


windowSpec = (
    Window
    .partitionBy("CustomerId", "date")
    .orderBy(F.desc("Quantity"))
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

maxPurchaseQuantity = F.max(F.col("Quantity")).over(windowSpec)

purchaseDenseRank = F.dense_rank().over(windowSpec)
purchaseRank = F.rank().over(windowSpec)

# Calculate
(
    dfWithDate
    .where("CustomerId IS NOT NULL")
    .orderBy("CustomerId")
    .select(
        F.col("CustomerId"),
        F.col("date"),
        F.col("Quantity"),
        purchaseRank.alias("quantityRank"),
        purchaseDenseRank.alias("quantityDenseRank"),
        maxPurchaseQuantity.alias("maxPurchaseQuantity")
    ).show()
)

+----------+----+--------+------------+-----------------+-------------------+
|CustomerId|date|Quantity|quantityRank|quantityDenseRank|maxPurchaseQuantity|
+----------+----+--------+------------+-----------------+-------------------+
|     12346|NULL|   74215|           1|                1|              74215|
|     12346|NULL|  -74215|           2|                2|              74215|
|     12347|NULL|     240|           1|                1|                240|
|     12347|NULL|      36|           2|                2|                240|
|     12347|NULL|      36|           2|                2|                240|
|     12347|NULL|      36|           2|                2|                240|
|     12347|NULL|      24|           5|                3|                240|
|     12347|NULL|      24|           5|                3|                240|
|     12347|NULL|      24|           5|                3|                240|
|     12347|NULL|      24|           5|                3|       

### Window functions PT
Imagina uma sala de aula. Um groupBy normal é tipo dividir a turma em grupos e cada grupo só te dá um resultado final (a média do grupo por exemplo) - perdes os alunos individuais.

Um window function é diferente: cada alguno mantém a sua linha própria, mas consegue espreitar para os colegas a volta dele (do mesmo grupo, ou numa certa ordem) e calcular algo baseado nisso - soma acumulada, ranking, média móvel, etc.. Ninguém desparece, só ganham "super-visão" dos vizinhos.

Uma window tem 3 peças:
* partitionBy → em que grupo (turma) estou
* orderBy → por que ordem olho para os colegas
* rowsBetween → quantos colegas para trás/frente eu consigo ver

In [24]:
# Vendas de 3 vendedores por mes
data = [
    ("Ana",   "Jan", 100),
    ("Ana",   "Fev", 150),
    ("Ana",   "Mar", 120),
    ("Ana",   "Abr", 200),
    ("Bruno", "Jan", 90),
    ("Bruno", "Fev", 90),
    ("Bruno", "Mar", 130),
    ("Bruno", "Abr", 110),
    ("Carla", "Jan", 300),
    ("Carla", "Fev", 250),
    ("Carla", "Mar", 400),
    ("Carla", "Abr", 350),
]

df = spark.createDataFrame(data, ["vendedor", "mes", "vendas"])

# Esta é a mais simples. Ordena por mês, dentro de cada vendedor
w = Window.partitionBy("vendedor").orderBy(F.col("mes"))

# Mostra-me para cada vendedor, o valor agrado tendo em conta a linha anterior. Ex;
# |     Ana|Abr|   200|            200|
# |     Ana|Fev|   150|            350| - Adicionou os 150
# |     Ana|Jan|   100|            450|
# |     Ana|Mar|   120|            570|
(
    df.withColumn("window_1", F.sum("vendas").over(w))
)

# Window.unboundedPreceding, Window.currentRow
# Do inicio até a linha atual
# Onde usar: total de vendas acumulado ao longo do ano, 
# saldo bancário corrido, contador de encomendas até à data.
# Mostra-me para cada vendedor, o valor agrado tendo em conta a linha anterior
# |     Ana|Abr|   200|            200|
# |     Ana|Fev|   150|            350|
# |     Ana|Jan|   100|            450|
# |     Ana|Mar|   120|            570|
w_running = w.rowsBetween(Window.unboundedPreceding, Window.currentRow)
(
    df.withColumn("total_acumulado", F.sum("vendas").over(w_running))
)

# Da linha atual até ao fim
# Inverte a ordem, mostra-me do maior para o mais pequeno
# Onde usar: "quanto é que ainda falta vender este ano a partir deste mês"
#  — útil para forecasts ou metas restantes.
# |     Ana|Abr|   200|              570| - o 570 é o total de vendas da Ana
# |     Ana|Fev|   150|              370| - Vai reduzindo o valor
# |     Ana|Jan|   100|              220|
# |     Ana|Mar|   120|              120|
w_restante = w.rowsBetween(Window.currentRow, Window.unboundedFollowing)

df.withColumn("vendas_ate_ao_fim", F.sum("vendas").over(w_restante))

# Só o vizinho anterior, o atual e o seguinte
# Onde usar: média móvel de 3 meses para suavizar picos, 
# muito comum em dashboards de tendência.
w_movel = w.rowsBetween(-1, 1)

df.withColumn("media_movel_3m", F.avg("vendas").over(w_movel))

# Onde usar: calcular o total do grupo mantendo o detalhe linha a linha 
# — ex: vendas / total_do_vendedor para dar a percentagem de contribuição de cada mês.
# |     Ana|Abr|   200|              570| 570 é o total de vendas da Ana
# |     Ana|Fev|   150|              570|
# |     Ana|Jan|   100|              570|
# |     Ana|Mar|   120|              570|
w_total = w.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

df.withColumn("total_do_vendedor", F.sum("vendas").over(w_total))

# -1, Window.currentRow — só o mês anterior + o atual (comparação simples com anterior)
# |     Ana|Abr|   200|         200| 200 - Valor do mes
# |     Ana|Fev|   150|         350| 350 - Valor do mes atual + mes anterior
# |     Ana|Jan|   100|         250| 250 - ....
# |     Ana|Mar|   120|         220|

# Onde usar: alternativa a F.lag() quando queres agregações 
# (não só o valor anterior, mas uma soma/média entre o atual e o anterior).
w_prev = w.rowsBetween(-1, Window.currentRow)

df.withColumn("soma_2_meses", F.sum("vendas").over(w_prev))

DataFrame[vendedor: string, mes: string, vendas: bigint, soma_2_meses: bigint]

In [ ]:
# Os ranks
# Window sem rowsBetween (rank olha para a ordenação toda dentro da partição):
# Vou rankar vendedores por mês, do maior para o menor vendedor.
w_rank = Window.partitionBy("mes").orderBy(F.col("vendas").desc())

# row_number() — número sequencial único, sem empates. 1, 2, 3, 4...
# Onde usar: quando precisas de exatamente 1 linha por grupo 
# (ex: "dá-me só a venda mais recente de cada vendedor" — filtras row_number == 1).
#  É o mais usado para deduplicação.
df.withColumn("row_number", F.row_number().over(w_rank))


# rank() — mesma posição para empates, mas salta números a seguir.
# Se dois empatam em 1º, o próximo é 3º.
# Onde usar: rankings tipo competição/olimpíadas — 
# se dois vendedores empatam em 1º lugar, 
# não faz sentido haver um "2º lugar", vai direto para o "3º".
df.withColumn("rank", F.rank().over(w_rank))


# dense_rank() — mesma posição para empates, mas não salta números.
# Empate em 1º, o próximo é 2º.
# Onde usar: quando queres saber "quantos níveis de performance distintos existem" 
# — ex: top 3 níveis de vendas (não top 3 vendedores), mantendo empates juntos 
# sem buracos na numeração.
df.withColumn("dense_rank", F.dense_rank().over(w_rank))


# percent_rank() — a posição relativa em percentagem (0 a 1).
# Onde usar: "este vendedor está no top 10% do mês?" 
# — bom para benchmarks e comparações relativas entre grupos de tamanhos diferentes.
df.withColumn("percent_rank", F.percent_rank().over(w_rank))


# ntile(n) — divide a partição em n grupos de tamanho igual (quartis, decis, etc).
# Onde usar: segmentação — ex: dividir clientes em 4 quartis por valor de compra 
# para campanhas de marketing diferentes por segmento.
df.withColumn("quartil", F.ntile(4).over(w_rank)).show()

+--------+---+------+-------+
|vendedor|mes|vendas|quartil|
+--------+---+------+-------+
|   Carla|Abr|   350|      1|
|     Ana|Abr|   200|      2|
|   Bruno|Abr|   110|      3|
|   Carla|Fev|   250|      1|
|     Ana|Fev|   150|      2|
|   Bruno|Fev|    90|      3|
|   Carla|Jan|   300|      1|
|     Ana|Jan|   100|      2|
|   Bruno|Jan|    90|      3|
|   Carla|Mar|   400|      1|
|   Bruno|Mar|   130|      2|
|     Ana|Mar|   120|      3|
+--------+---+------+-------+



## Grouping sets

In [8]:
dfNotNull = dfWithDate.drop()
dfNotNull.createOrReplaceTempView("dfNotNull")

In [11]:
spark.sql(
    """
    select CustomerId, stockCode, sum(Quantity) from dfNotNull
    group by customerId, stockCode
    order by CustomerId DESC, stockCode DESc
    """
).show(10)

# We can also use grouping sets
spark.sql(
    """
    select CustomerId, stockCode, sum(Quantity) from dfNotNull
    group by customerId, stockCode grouping sets((customerId, stockCode), ())
    order by CustomerId DESC, stockCode DESc
    """
).show(10)

+----------+---------+-------------+
|CustomerId|stockCode|sum(Quantity)|
+----------+---------+-------------+
|     18287|    85173|           48|
|     18287|   85040A|           48|
|     18287|   85039B|          120|
|     18287|   85039A|           96|
|     18287|    84920|            4|
|     18287|    84584|            6|
|     18287|   84507C|            6|
|     18287|   72351B|           24|
|     18287|   72351A|           24|
|     18287|   72349B|           60|
+----------+---------+-------------+
only showing top 10 rows


+----------+---------+-------------+
|customerId|stockCode|sum(Quantity)|
+----------+---------+-------------+
|     18287|    85173|           48|
|     18287|   85040A|           48|
|     18287|   85039B|          120|
|     18287|   85039A|           96|
|     18287|    84920|            4|
|     18287|    84584|            6|
|     18287|   84507C|            6|
|     18287|   72351B|           24|
|     18287|   72351A|           24|
|     18287|   72349B|           60|
+----------+---------+-------------+
only showing top 10 rows


## Rollups

In [12]:
rolledUpDF = (
    dfNotNull.rollup("Date", "Country").agg(F.sum("Quantity"))
    .selectExpr("Date", "Country", "`sum(Quantity)`as total_quantity")
    .orderBy("Date")
)
rolledUpDF.show()

26/10/02 20:46:24 ERROR Executor: Exception in task 2.0 in stage 13.0 (TID 49)
org.apache.spark.SparkDateTimeException: [CANNOT_PARSE_TIMESTAMP] Text '6/23/2011 15:35' could not be parsed at index 0. Use `try_to_date` to tolerate invalid input string and return NULL instead. SQLSTATE: 22007
	at org.apache.spark.sql.errors.QueryExecutionErrors$.ansiDateTimeParseError(QueryExecutionErrors.scala:290)
	at org.apache.spark.sql.errors.QueryExecutionErrors.ansiDateTimeParseError(QueryExecutionErrors.scala)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.hashAgg_doAggregateWithKeys_0$(Unknown Source)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:44)
	at org.apache.spark.sql.execution.WholeStageCodegenEvaluatorFactory$WholeStageCodegenPartitionEvaluator$$anon$1.hasNext(WholeStageCodegen

DateTimeException: [CANNOT_PARSE_TIMESTAMP] Text '6/23/2011 15:35' could not be parsed at index 0. Use `try_to_date` to tolerate invalid input string and return NULL instead. SQLSTATE: 22007